# Cuaderno ML 2: Predicción del crecimiento de jugadores con Random Forest

**Institución Universitaria Pascual Bravo**
**Asignatura:** Gestión de Datos Masivos
**Docente:** Genry Gutiérrez
**Proyecto:** Steam Game Intelligence
**Infraestructura:** Databricks Lakehouse, Unity Catalog, Delta Lake, Spark MLlib y MLflow
**Fuente:** capa Gold (`game_growth`, `dim_game`, `twitch_steam_relationship`)

---

## Pregunta de negocio

> **¿Podemos predecir si un juego crecerá al menos un 10 % en jugadores el mes siguiente? ¿La exposición en Twitch mejora esa predicción?**

La pregunta original del proyecto es si la exposición en Twitch se asocia con cambios posteriores en Steam. La correlación descriptiva fue casi nula (−0,0032), pero una correlación lineal no detecta relaciones no lineales ni interacciones. Este cuaderno la replantea como un problema de predicción: se entrena el mismo modelo **con y sin** las variables de Twitch y se mide si la capacidad predictiva mejora.

## Objetivos

1. Construir desde Gold una tabla juego-mes con variables conocidas en el mes *t* y el objetivo en *t+1*.
2. Evitar la fuga de información con variables solo del pasado y una partición temporal.
3. Entrenar un **Random Forest** con Spark MLlib, con pesos por clase para el desbalance.
4. Evaluar con métricas adecuadas a una clase minoritaria (PR-AUC, precisión y recall de la clase positiva) frente a un modelo de referencia.
5. Interpretar la importancia de variables, registrar los modelos en MLflow y guardar las predicciones en Gold.

---
## 1. Fundamentos: aprendizaje supervisado

Se dispone de un conjunto etiquetado

$$\mathcal{D} = \{(\mathbf{x}_1, y_1), \dots, (\mathbf{x}_N, y_N)\}$$

- $\mathbf{x}_i$: variables conocidas del juego en el mes *t* (jugadores, tendencia, ciclo de vida, catálogo, Twitch).
- $y_i \in \{0, 1\}$: **1** si el juego creció al menos un 10 % en *t+1*, **0** en otro caso.

El objetivo es encontrar una función $\hat{f}(\mathbf{x})$ que acierte en datos **futuros** que el modelo no vio.

| Tipo | Variable objetivo | Ejemplo en este proyecto |
| :--- | :--- | :--- |
| **Clasificación binaria (este cuaderno)** | $\{0, 1\}$ | ¿El juego crecerá ≥ 10 % el próximo mes? |
| Regresión | Continua | ¿Cuántos jugadores tendrá el próximo mes? |

## 2. Del árbol de decisión al Random Forest

Un **árbol de decisión** divide los datos con preguntas sucesivas, por ejemplo: *¿el juego está a menos del 30 % de su pico? → ¿el mes siguiente hay rebajas de Steam? → ¿está en el top 200 de Twitch?* Cada división busca nodos lo más puros posible según la **impureza de Gini**:

$$I_G = 1 - \sum_{c} p_c^2$$

Un árbol solo tiende a **sobreajustarse**: memoriza los datos de entrenamiento y falla con datos nuevos. El **Random Forest** entrena muchos árboles distintos y promedia sus votos. Dos fuentes de azar hacen que los árboles se equivoquen de forma diferente y el promedio sea estable:

1. **Bagging:** cada árbol se entrena con una muestra aleatoria de las filas.
2. **Submuestreo de variables:** en cada división, el árbol solo puede elegir entre un subconjunto aleatorio de variables ($\sqrt{p}$).

### Por qué Random Forest para estos datos

- Los datos son **tabulares**, con escalas muy distintas (jugadores, meses, ratios, rankings). Los árboles no necesitan estandarización.
- Las relaciones esperadas **no son lineales**: el efecto de las rebajas depende de la etapa del ciclo de vida, y estar en el top 10 de Twitch puede importar mucho más que estar en el puesto 190.
- Entrega la **importancia de cada variable**, lo que permite ver cuánto pesa Twitch dentro del modelo.

---
## 3. Desbalance de clases y fuga de información

### Desbalance

Crecer un 10 % en un mes es poco frecuente. Si solo una minoría de las filas es positiva, un modelo que siempre responda "no crece" tendría una exactitud alta y no serviría para nada. Por eso:

- La métrica principal es **PR-AUC** (área bajo la curva precisión-recall), que se concentra en la clase positiva. Su valor de referencia es la tasa de positivos: un modelo al azar obtiene un PR-AUC igual a esa tasa.
- Se reportan **precisión, recall y F1 de la clase positiva**, no las versiones ponderadas. Las ponderadas mezclan las dos clases y ocultan cómo le va al modelo con la minoritaria.
- Se entrena con **pesos por clase** (`weightCol`): cada fila positiva pesa más, de modo que las dos clases aportan lo mismo al entrenamiento.

### Fuga de información (data leakage)

Un modelo que "ve el futuro" da métricas excelentes en el notebook y falla en la realidad. Se evita con tres reglas:

1. **Todas las variables se calculan con datos hasta el mes *t*.** Por ejemplo, la distancia al pico usa el máximo histórico *hasta t*, no el máximo de toda la vida del juego.
2. **Se excluyen** `positive`, `negative`, `review_ratio` y `owners_*` de `dim_game`: son una foto tomada en 2026 y reflejan el éxito futuro del juego.
3. **Partición temporal:** se entrena con 2016–2021 y se evalúa con 2022–2024. Una partición aleatoria mezclaría meses futuros en el entrenamiento.

---
## 4. Configuración

In [0]:
%pip install plotly --quiet

In [0]:
import math
import os
import re
from datetime import datetime, timezone

import mlflow
import mlflow.spark
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.functions import vector_to_array
from pyspark.sql import Window
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "steam_game_intelligence")
dbutils.widgets.text("gold_schema", "gold")
dbutils.widgets.text("min_players", "20")
dbutils.widgets.text("growth_threshold", "0.10")
dbutils.widgets.text("window_start", "2016-01-01")
dbutils.widgets.text("window_end", "2024-08-01")
dbutils.widgets.text("train_end", "2021-12-01")
dbutils.widgets.text("num_trees", "100")
dbutils.widgets.text("max_depth", "8")
dbutils.widgets.text("top_genres", "8")
dbutils.widgets.dropdown("persist_results", "true", ["true", "false"])

catalog = dbutils.widgets.get("catalog").strip()
gold_schema = dbutils.widgets.get("gold_schema").strip()
MIN_PLAYERS = float(dbutils.widgets.get("min_players"))
THRESHOLD = float(dbutils.widgets.get("growth_threshold"))
WINDOW_START = dbutils.widgets.get("window_start").strip()
WINDOW_END = dbutils.widgets.get("window_end").strip()
TRAIN_END = dbutils.widgets.get("train_end").strip()
NUM_TREES = int(dbutils.widgets.get("num_trees"))
MAX_DEPTH = int(dbutils.widgets.get("max_depth"))
TOP_GENRES = int(dbutils.widgets.get("top_genres"))
PERSIST = dbutils.widgets.get("persist_results") == "true"

if not (WINDOW_START < TRAIN_END < WINDOW_END):
    raise ValueError("Se requiere window_start < train_end < window_end (formato YYYY-MM-DD)")

SEED = 42
OUTPUT_TABLE = "player_growth_predictions"


def gold_table(name):
    return f"`{catalog}`.`{gold_schema}`.`{name}`"


# Tablas que se exportan al final en Markdown para revisar los resultados fuera de Databricks.
REPORT = []


def add_report(title, pdf):
    REPORT.append((title, pdf.copy()))
    return pdf


print(
    f"Gold: {catalog}.{gold_schema} | umbral de crecimiento={THRESHOLD:.0%} | min_players={MIN_PLAYERS} | "
    f"ventana {WINDOW_START} a {WINDOW_END} | train hasta {TRAIN_END}"
)

---
## 5. Carga de la capa Gold

In [0]:
game_growth = spark.table(gold_table("game_growth"))
dim_game = spark.table(gold_table("dim_game"))
twitch = spark.table(gold_table("twitch_steam_relationship"))

volumetria = pd.DataFrame([
    {"tabla": "game_growth", "filas": game_growth.count(), "juegos": game_growth.select("app_id").distinct().count()},
    {"tabla": "dim_game", "filas": dim_game.count(), "juegos": dim_game.select("app_id").distinct().count()},
    {"tabla": "twitch_steam_relationship", "filas": twitch.count(), "juegos": twitch.select("app_id").distinct().count()},
])
display(add_report("Volumetría de las tablas Gold", volumetria))

---
## 6. Ingeniería de variables y definición del objetivo

Gold no trae el valor del mes siguiente, así que se calcula con ventanas de Spark (`lag`, `lead` y máximo acumulado) sobre la historia completa de cada juego. Solo después se recorta a la ventana de análisis, para que los primeros meses de la ventana tengan su historia previa.

**Objetivo:**

```text
target = 1  si  avg_players(t+1) >= (1 + umbral) × avg_players(t)   y t+1 es el mes siguiente a t
target = 0  en otro caso
```

Las filas sin un mes *t+1* consecutivo no tienen objetivo y se descartan.

| Grupo | Variable | Cálculo (siempre con datos hasta *t*) |
|---|---|---|
| Dinámica | `log_avg_players` | `log1p(avg_players)` |
| | `log_change_1m` | Cambio en logaritmo respecto al mes anterior (0 si no hay mes anterior consecutivo) |
| | `log_change_3m` | Cambio en logaritmo respecto a 3 meses antes (0 si no existe) |
| | `ratio_to_ma3` | `avg_players` ÷ media móvil de 3 meses |
| | `drawdown_from_peak` | `avg_players` ÷ máximo histórico **hasta t** |
| | `peak_to_avg_ratio` | Pico ÷ promedio dentro del mes |
| Ciclo de vida y calendario | `months_since_release` | Meses desde el lanzamiento (−1 si no hay fecha) |
| | `next_month` | Mes calendario de *t+1* (1–12) |
| | `next_month_is_steam_sale` | 1 si *t+1* es junio, julio o diciembre (rebajas de verano e invierno de Steam). El calendario del mes siguiente se conoce en *t*, así que no es fuga |
| Catálogo | `is_free`, `log_price` | Precio en `dim_game` |
| | `genre_*` | Indicadores de los géneros más frecuentes en entrenamiento |
| Twitch | `in_twitch_top200` | 1 si el juego está en el top 200 de Twitch en *t* |
| | `log_hours_watched` | `log1p(horas vistas en t)`, 0 si no está |
| | `twitch_rank_filled` | Posición en el ranking en *t*, 201 si no está |

Las variables `has_*` indican cuándo un valor fue imputado (por ejemplo, `has_release_date = 0` cuando `months_since_release = −1`), para que el modelo distinga un valor real de uno faltante.

In [0]:
w = Window.partitionBy("app_id").orderBy("month_start")
w_cumulative = w.rowsBetween(Window.unboundedPreceding, Window.currentRow)

panel = (
    game_growth
    .filter(F.col("avg_players").isNotNull())
    .select(
        "app_id",
        "name",
        "month_start",
        F.col("avg_players").cast("double").alias("avg_players"),
        F.col("peak_to_avg_ratio").cast("double").alias("peak_to_avg_ratio_raw"),
        F.col("avg_players_ma3").cast("double").alias("avg_players_ma3"),
        F.col("months_since_release").cast("double").alias("months_since_release_raw"),
    )
    .withColumn("prev_month_start", F.lag("month_start").over(w))
    .withColumn("prev_avg_players", F.lag("avg_players").over(w))
    .withColumn("lag3_month_start", F.lag("month_start", 3).over(w))
    .withColumn("lag3_avg_players", F.lag("avg_players", 3).over(w))
    .withColumn("next_month_start", F.lead("month_start").over(w))
    .withColumn("next_avg_players", F.lead("avg_players").over(w))
    .withColumn("hist_max_players", F.max("avg_players").over(w_cumulative))
)

has_prev = F.coalesce(F.col("prev_month_start") == F.add_months("month_start", -1), F.lit(False))
has_lag3 = F.coalesce(F.col("lag3_month_start") == F.add_months("month_start", -3), F.lit(False))
has_next = F.coalesce(F.col("next_month_start") == F.add_months("month_start", 1), F.lit(False))

features = (
    panel.filter(has_next)
    .withColumn("growth_next", F.expr("try_divide(next_avg_players - avg_players, avg_players)"))
    .withColumn("target", F.when(F.col("next_avg_players") >= (1 + THRESHOLD) * F.col("avg_players"), 1.0).otherwise(0.0))
    # Dinámica
    .withColumn("log_avg_players", F.log1p("avg_players"))
    .withColumn("has_prev_month", has_prev.cast("double"))
    .withColumn("log_change_1m", F.when(has_prev, F.log1p("avg_players") - F.log1p("prev_avg_players")).otherwise(0.0))
    .withColumn("has_lag_3m", has_lag3.cast("double"))
    .withColumn("log_change_3m", F.when(has_lag3, F.log1p("avg_players") - F.log1p("lag3_avg_players")).otherwise(0.0))
    .withColumn("ratio_to_ma3", F.coalesce(F.expr("try_divide(avg_players, avg_players_ma3)"), F.lit(1.0)))
    .withColumn("drawdown_from_peak", F.coalesce(F.expr("try_divide(avg_players, hist_max_players)"), F.lit(1.0)))
    .withColumn("peak_to_avg_ratio", F.coalesce(F.col("peak_to_avg_ratio_raw"), F.lit(1.0)))
    # Ciclo de vida y calendario
    .withColumn("has_release_date", F.col("months_since_release_raw").isNotNull().cast("double"))
    .withColumn("months_since_release", F.coalesce(F.col("months_since_release_raw"), F.lit(-1.0)))
    .withColumn("next_month", F.month(F.add_months("month_start", 1)).cast("double"))
    .withColumn("next_month_is_steam_sale", F.col("next_month").isin(6.0, 7.0, 12.0).cast("double"))
    # Referencia: "crece si creció este mes"
    .withColumn(
        "baseline_pred",
        F.when(has_prev & (F.col("avg_players") >= (1 + THRESHOLD) * F.col("prev_avg_players")), 1.0).otherwise(0.0),
    )
)

catalog_info = dim_game.select(
    "app_id",
    F.col("price").cast("double").alias("price"),
    "genres",
    F.lit(1.0).alias("has_catalog_info"),
)
twitch_month = twitch.select(
    "app_id",
    "month_start",
    F.col("twitch_rank").cast("double").alias("twitch_rank"),
    F.col("hours_watched").cast("double").alias("hours_watched"),
)

features = (
    features
    .join(catalog_info, "app_id", "left")
    .withColumn("has_catalog_info", F.coalesce(F.col("has_catalog_info"), F.lit(0.0)))
    .withColumn("is_free", F.when(F.col("price") == 0, 1.0).otherwise(0.0))
    .withColumn("log_price", F.log1p(F.coalesce(F.col("price"), F.lit(0.0))))
    .join(twitch_month, ["app_id", "month_start"], "left")
    .withColumn("in_twitch_top200", F.col("twitch_rank").isNotNull().cast("double"))
    .withColumn("log_hours_watched", F.log1p(F.coalesce(F.col("hours_watched"), F.lit(0.0))))
    .withColumn("twitch_rank_filled", F.coalesce(F.col("twitch_rank"), F.lit(201.0)))
)

dataset = (
    features
    .filter(F.col("month_start").between(F.to_date(F.lit(WINDOW_START)), F.to_date(F.lit(WINDOW_END))))
    .filter(F.col("avg_players") >= MIN_PLAYERS)
    .cache()
)
print(f"Filas juego-mes en la ventana: {dataset.count():,} | juegos: {dataset.select('app_id').distinct().count():,}")

### 6.1 Fase 0: validación del umbral de crecimiento

Antes de fijar el 10 % se revisa cuántas filas serían positivas con distintos umbrales y cómo se distribuye el crecimiento mensual. Un umbral muy bajo convierte en "crecimiento" el ruido normal; uno muy alto deja muy pocos positivos para aprender.

In [0]:
threshold_options = [0.05, 0.10, 0.20, 0.30]
rate_row = dataset.agg(*[
    F.avg(F.when(F.col("growth_next") >= t, 1.0).otherwise(0.0)).alias(f"t{int(t * 100)}") for t in threshold_options
]).first()
threshold_rates = pd.DataFrame([
    {"umbral_crecimiento": f"{t:.0%}", "pct_filas_positivas": round(100 * rate_row[f"t{int(t * 100)}"], 2)}
    for t in threshold_options
])
display(add_report("Tasa de positivos según el umbral de crecimiento", threshold_rates))

bin_width = 0.05
growth_hist = (
    dataset
    .withColumn("growth_clipped", F.greatest(F.lit(-0.5), F.least(F.lit(1.0), F.col("growth_next"))))
    .withColumn("bin", F.round(F.floor(F.col("growth_clipped") / bin_width) * bin_width, 2))
    .groupBy("bin").count().orderBy("bin").toPandas()
)
fig = px.bar(
    growth_hist,
    x="bin",
    y="count",
    title="Distribución del crecimiento mensual t → t+1 (recortado a [−50 %, +100 %])",
    labels={"bin": "Crecimiento relativo", "count": "Filas juego-mes"},
    template="plotly_white",
)
fig.add_vline(x=THRESHOLD, line_dash="dash", line_color="red", annotation_text=f"umbral {THRESHOLD:.0%}")
fig.update_layout(height=440, xaxis_tickformat=".0%")
fig.show()

### 6.2 Géneros y lista final de variables

Los géneros se eligen con los juegos del **periodo de entrenamiento** solamente. Así ninguna decisión de preparación usa información del periodo de prueba.

In [0]:
train_games = dataset.filter(F.col("month_start") <= F.to_date(F.lit(TRAIN_END))).select("app_id").distinct()
top_genres = [
    row["genre"]
    for row in (
        train_games.join(catalog_info.select("app_id", "genres"), "app_id")
        .select(F.explode("genres").alias("genre"))
        .groupBy("genre").count()
        .orderBy(F.desc("count"), "genre")
        .limit(TOP_GENRES)
        .collect()
    )
]


def genre_column(genre):
    return "genre_" + re.sub(r"[^a-z0-9]+", "_", genre.lower()).strip("_")


GENRE_COLUMNS = {genre: genre_column(genre) for genre in top_genres}
for genre, column in GENRE_COLUMNS.items():
    dataset = dataset.withColumn(column, F.coalesce(F.array_contains("genres", genre).cast("double"), F.lit(0.0)))

FEATURES_BASE = [
    # Dinámica
    "log_avg_players", "log_change_1m", "has_prev_month", "log_change_3m", "has_lag_3m",
    "ratio_to_ma3", "drawdown_from_peak", "peak_to_avg_ratio",
    # Ciclo de vida y calendario
    "months_since_release", "has_release_date", "next_month", "next_month_is_steam_sale",
    # Catálogo
    "is_free", "log_price", "has_catalog_info",
    *GENRE_COLUMNS.values(),
]
FEATURES_TWITCH = ["in_twitch_top200", "log_hours_watched", "twitch_rank_filled"]
FEATURES_ALL = FEATURES_BASE + FEATURES_TWITCH

print(f"Géneros seleccionados: {top_genres}")
print(f"Variables sin Twitch: {len(FEATURES_BASE)} | con Twitch: {len(FEATURES_ALL)}")

---
## 7. Partición temporal y pesos por clase

| Conjunto | Meses *t* | Uso |
|---|---|---|
| Train | `window_start` a `train_end` | Entrenamiento |
| Test | después de `train_end` hasta `window_end` | Evaluación en el "futuro" |

El peso de cada clase se calcula solo con train: $w_1 = \frac{0{,}5}{p}$ y $w_0 = \frac{0{,}5}{1-p}$, donde $p$ es la tasa de positivos. Así las dos clases suman el mismo peso total.

In [0]:
model_df = dataset.select(
    "app_id", "name", "month_start", "avg_players", "next_avg_players", "growth_next", "target", "baseline_pred",
    *FEATURES_ALL,
)

train_df = model_df.filter(F.col("month_start") <= F.to_date(F.lit(TRAIN_END)))
test_df = model_df.filter(F.col("month_start") > F.to_date(F.lit(TRAIN_END))).cache()

positive_rate_train = train_df.agg(F.avg("target")).first()[0]
weight_positive = 0.5 / positive_rate_train
weight_negative = 0.5 / (1 - positive_rate_train)
train_df = train_df.withColumn(
    "class_weight", F.when(F.col("target") == 1.0, weight_positive).otherwise(weight_negative)
).cache()

balance = (
    train_df.withColumn("conjunto", F.lit("train")).select("conjunto", "target", "app_id")
    .unionByName(test_df.withColumn("conjunto", F.lit("test")).select("conjunto", "target", "app_id"))
    .groupBy("conjunto")
    .agg(
        F.count("*").alias("filas"),
        F.countDistinct("app_id").alias("juegos"),
        F.sum("target").cast("long").alias("positivos"),
        F.round(100 * F.avg("target"), 2).alias("pct_positivos"),
    )
    .orderBy(F.desc("conjunto"))
    .toPandas()
)
positive_rate_test = float(balance.loc[balance["conjunto"] == "test", "pct_positivos"].iloc[0]) / 100
display(add_report("Balance de clases por conjunto", balance))
print(f"Pesos de clase: positivo={weight_positive:.3f} | negativo={weight_negative:.3f}")

fig = px.bar(
    balance,
    x="conjunto",
    y="pct_positivos",
    text="pct_positivos",
    title=f"% de filas positivas (crecimiento ≥ {THRESHOLD:.0%}) por conjunto",
    labels={"pct_positivos": "% positivos", "conjunto": ""},
    template="plotly_white",
)
fig.update_layout(height=400)
fig.show()

---
## 8. Pipeline y entrenamiento con MLflow

```
[variables numéricas e indicadores]
                │
                ▼
VectorAssembler → 'features'
                │
                ▼
RandomForestClassifier (weightCol = class_weight)
```

No hay `StringIndexer` ni `OneHotEncoder`: los géneros ya son indicadores 0/1 y el resto de variables es numérico. Tampoco hay `StandardScaler`, porque los árboles dividen por umbrales y no dependen de la escala.

Se comparan tres modelos sobre **las mismas filas de test**:

| Modelo | Qué es | Para qué sirve |
|---|---|---|
| Referencia | "Crece si creció este mes" | Piso mínimo: un modelo que no lo supere no aporta valor |
| RF sin Twitch | Random Forest con dinámica, ciclo de vida y catálogo | Modelo base |
| RF con Twitch | El mismo modelo + 3 variables de Twitch | Responde la pregunta de negocio |

La métrica PR-AUC de la referencia se calcula con su puntaje `log_change_1m` (cuánto creció este mes), que ordena las filas de forma continua.

In [0]:
VOLUME_TMP = f"/Volumes/{catalog}/{gold_schema}/mlflow_tmp"
try:
    spark.sql(f"CREATE VOLUME IF NOT EXISTS `{catalog}`.`{gold_schema}`.`mlflow_tmp`")
    os.environ["MLFLOW_DFS_TMP"] = VOLUME_TMP
    print(f"Volumen para MLflow: {VOLUME_TMP}")
except Exception as error:
    print(f"Aviso sobre el volumen de MLflow: {error}")


def evaluate(df, model_name, score_col, prediction_col):
    """Métricas de la clase positiva + PR-AUC y ROC-AUC. Devuelve (métricas, matriz de confusión)."""
    pr_auc = BinaryClassificationEvaluator(labelCol="target", rawPredictionCol=score_col, metricName="areaUnderPR").evaluate(df)
    roc_auc = BinaryClassificationEvaluator(labelCol="target", rawPredictionCol=score_col, metricName="areaUnderROC").evaluate(df)
    counts = {(int(r["target"]), int(r[prediction_col])): r["count"] for r in df.groupBy("target", prediction_col).count().collect()}
    tp, fp = counts.get((1, 1), 0), counts.get((0, 1), 0)
    fn, tn = counts.get((1, 0), 0), counts.get((0, 0), 0)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    metrics = {
        "modelo": model_name,
        "pr_auc": round(pr_auc, 4),
        "roc_auc": round(roc_auc, 4),
        "precision_pos": round(precision, 4),
        "recall_pos": round(recall, 4),
        "f1_pos": round(f1, 4),
        "accuracy": round((tp + tn) / max(tp + fp + fn + tn, 1), 4),
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
    }
    return metrics, counts


def build_pipeline(feature_cols):
    assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
    forest = RandomForestClassifier(
        labelCol="target",
        featuresCol="features",
        weightCol="class_weight",
        numTrees=NUM_TREES,
        maxDepth=MAX_DEPTH,
        subsamplingRate=0.8,
        featureSubsetStrategy="sqrt",
        seed=SEED,
    )
    return Pipeline(stages=[assembler, forest])


results = []
confusions = {}
predictions = {}
fitted_models = {}
run_ids = {}

baseline_scored = test_df.withColumn("baseline_score", F.col("log_change_1m"))
baseline_metrics, confusions["referencia"] = evaluate(baseline_scored, "referencia", "baseline_score", "baseline_pred")
results.append(baseline_metrics)

experiments = [("rf_sin_twitch", FEATURES_BASE), ("rf_con_twitch", FEATURES_ALL)]
for model_name, feature_cols in experiments:
    with mlflow.start_run(run_name=f"steam_growth_{model_name}") as run:
        run_ids[model_name] = run.info.run_id
        mlflow.log_params({
            "algoritmo": "RandomForestClassifier",
            "variante": model_name,
            "num_trees": NUM_TREES,
            "max_depth": MAX_DEPTH,
            "growth_threshold": THRESHOLD,
            "min_players": MIN_PLAYERS,
            "train": f"{WINDOW_START} a {TRAIN_END}",
            "test": f"despues de {TRAIN_END} a {WINDOW_END}",
            "n_variables": len(feature_cols),
        })
        mlflow.log_param("variables", ",".join(feature_cols)[:5000])

        print(f"Entrenando {model_name} con {len(feature_cols)} variables...")
        model = build_pipeline(feature_cols).fit(train_df)
        scored = model.transform(test_df).withColumn("probability_growth", vector_to_array("probability")[1]).cache()

        metrics, confusions[model_name] = evaluate(scored, model_name, "probability", "prediction")
        mlflow.log_metrics({k: float(v) for k, v in metrics.items() if k != "modelo"})
        try:
            mlflow.spark.log_model(model, f"modelo_{model_name}", dfs_tmpdir=VOLUME_TMP)
        except Exception as error:
            print(f"Aviso al guardar el modelo en MLflow (las métricas sí se registraron): {error}")

        results.append(metrics)
        predictions[model_name] = scored
        fitted_models[model_name] = model
        print(f"  PR-AUC={metrics['pr_auc']:.4f} | ROC-AUC={metrics['roc_auc']:.4f} | recall={metrics['recall_pos']:.4f} | precisión={metrics['precision_pos']:.4f}")

---
## 9. Evaluación en el periodo de prueba

In [0]:
results_pd = pd.DataFrame(results)
display(add_report("Métricas en test (clase positiva = crecimiento ≥ umbral)", results_pd))
print(f"PR-AUC de un modelo al azar (tasa de positivos en test): {positive_rate_test:.4f}")

delta = results_pd.set_index("modelo")
pr_gain = delta.loc["rf_con_twitch", "pr_auc"] - delta.loc["rf_sin_twitch", "pr_auc"]
print(f"Diferencia de PR-AUC con Twitch vs. sin Twitch: {pr_gain:+.4f}")

metrics_long = results_pd.melt(
    id_vars="modelo", value_vars=["pr_auc", "roc_auc", "precision_pos", "recall_pos", "f1_pos"],
    var_name="metrica", value_name="valor",
)
fig = px.bar(
    metrics_long,
    x="metrica",
    y="valor",
    color="modelo",
    barmode="group",
    text_auto=".3f",
    title="Comparación de modelos en test",
    labels={"metrica": "", "valor": "Valor", "modelo": "Modelo"},
    template="plotly_white",
)
fig.add_hline(y=positive_rate_test, line_dash="dot", line_color="gray", annotation_text="PR-AUC al azar")
fig.update_layout(height=480)
fig.show()

### 9.1 Curvas precisión-recall y matriz de confusión

La curva precisión-recall muestra el compromiso del modelo: si se baja el umbral de decisión se detectan más juegos que van a crecer (más recall), a costa de más falsas alarmas (menos precisión).

In [0]:
def pr_curve(y_true, score, max_points=400):
    order = np.argsort(-score, kind="mergesort")
    y_sorted = y_true[order]
    tp = np.cumsum(y_sorted)
    fp = np.cumsum(1 - y_sorted)
    precision = tp / (tp + fp)
    recall = tp / max(y_sorted.sum(), 1)
    step = max(len(order) // max_points, 1)
    return recall[::step], precision[::step]


curves = []
baseline_pd = baseline_scored.select("target", "baseline_score").toPandas()
r, p = pr_curve(baseline_pd["target"].to_numpy(), baseline_pd["baseline_score"].to_numpy())
curves.append(pd.DataFrame({"recall": r, "precision": p, "modelo": "referencia"}))
for model_name, scored in predictions.items():
    pdf = scored.select("target", "probability_growth").toPandas()
    r, p = pr_curve(pdf["target"].to_numpy(), pdf["probability_growth"].to_numpy())
    curves.append(pd.DataFrame({"recall": r, "precision": p, "modelo": model_name}))

fig = px.line(
    pd.concat(curves),
    x="recall",
    y="precision",
    color="modelo",
    title="Curvas precisión-recall en test",
    labels={"recall": "Recall (juegos que crecen detectados)", "precision": "Precisión (alertas correctas)", "modelo": "Modelo"},
    template="plotly_white",
)
fig.add_hline(y=positive_rate_test, line_dash="dot", line_color="gray", annotation_text="azar")
fig.update_layout(height=500, yaxis_range=[0, 1])
fig.show()

best_model_name = delta.loc[["rf_sin_twitch", "rf_con_twitch"], "pr_auc"].idxmax()
cm = confusions[best_model_name]
cm_matrix = pd.DataFrame(
    [[cm.get((0, 0), 0), cm.get((0, 1), 0)], [cm.get((1, 0), 0), cm.get((1, 1), 0)]],
    index=["Real: no crece", "Real: crece"],
    columns=["Predicho: no crece", "Predicho: crece"],
)
add_report(f"Matriz de confusión en test ({best_model_name})", cm_matrix.reset_index().rename(columns={"index": ""}))
fig = px.imshow(
    cm_matrix,
    text_auto=",",
    color_continuous_scale="Blues",
    title=f"Matriz de confusión en test — {best_model_name}",
    labels={"color": "Filas"},
)
fig.update_layout(template="plotly_white", height=420)
fig.show()

### 9.2 ¿Twitch ayuda donde hay exposición?

En la mayoría de las filas el juego no está en el top 200 de Twitch, así que las variables de Twitch valen cero y no pueden cambiar la predicción. Para ver su efecto real se repite la evaluación **solo con las filas de test en las que el juego sí estaba en el top 200** ese mes.

In [0]:
twitch_subset_rows = []
subset_filter = F.col("in_twitch_top200") == 1.0
subset_metrics, _ = evaluate(baseline_scored.filter(subset_filter), "referencia", "baseline_score", "baseline_pred")
twitch_subset_rows.append(subset_metrics)
for model_name, scored in predictions.items():
    subset_metrics, _ = evaluate(scored.filter(subset_filter), model_name, "probability", "prediction")
    twitch_subset_rows.append(subset_metrics)

twitch_subset = pd.DataFrame(twitch_subset_rows)
n_subset = int(twitch_subset.loc[0, ["tp", "fp", "fn", "tn"]].sum())
display(add_report(f"Métricas en test solo con filas en el top 200 de Twitch ({n_subset:,} filas)", twitch_subset))

---
## 10. Interpretabilidad: ¿qué variables usa el modelo?

La importancia de Gini mide cuánto reduce la impureza cada variable, sumado en todas las divisiones de todos los árboles. Es relativa (suma 1) y tiene un sesgo conocido: favorece a las variables continuas con muchos valores posibles. Por eso se complementa con la comparación con y sin Twitch, que mide el aporte real en test.

In [0]:
importance_frames = []
for model_name, feature_cols in experiments:
    forest_model = fitted_models[model_name].stages[-1]
    importance_frames.append(pd.DataFrame({
        "modelo": model_name,
        "variable": feature_cols,
        "importancia": forest_model.featureImportances.toArray(),
    }))
importances = pd.concat(importance_frames)
importances["es_twitch"] = importances["variable"].isin(FEATURES_TWITCH)

top_twitch_model = importances[importances["modelo"] == "rf_con_twitch"].sort_values("importancia", ascending=False)
display(add_report("Importancia de variables (rf_con_twitch)", top_twitch_model.round(4)))
twitch_share = top_twitch_model.loc[top_twitch_model["es_twitch"], "importancia"].sum()
print(f"Importancia total de las variables de Twitch: {twitch_share:.2%}")

fig = px.bar(
    top_twitch_model.head(15),
    x="importancia",
    y="variable",
    color="es_twitch",
    orientation="h",
    title="Top 15 variables del Random Forest con Twitch (importancia de Gini)",
    labels={"importancia": "Importancia relativa", "variable": "", "es_twitch": "Variable de Twitch"},
    color_discrete_map={True: "#9146FF", False: "#1b9e77"},
    template="plotly_white",
)
fig.update_layout(yaxis=dict(autorange="reversed"), height=520)
fig.show()

---
## 11. Escenarios simulados

Se pasan por el modelo con Twitch cinco juegos hipotéticos con perfiles contrastantes. Sirve para explicar en la sustentación cómo reacciona el modelo, igual que los escenarios del ejemplo de clase.

In [0]:
def genre_flag(genre):
    return {GENRE_COLUMNS[genre]: 1.0} if genre in GENRE_COLUMNS else {}


def scenario(description, **values):
    row = {column: 0.0 for column in FEATURES_ALL}
    row.update({
        "has_prev_month": 1.0, "has_lag_3m": 1.0, "has_release_date": 1.0, "has_catalog_info": 1.0,
        "ratio_to_ma3": 1.0, "drawdown_from_peak": 1.0, "peak_to_avg_ratio": 1.3,
        "twitch_rank_filled": 201.0, "log_price": math.log1p(19.99), "next_month": 3.0,
    })
    row.update(values)
    row["next_month_is_steam_sale"] = 1.0 if row["next_month"] in (6.0, 7.0, 12.0) else 0.0
    row["escenario"] = description
    return row


scenarios = [
    scenario(
        "Juego mediano en caída, antes de las rebajas de verano",
        log_avg_players=math.log1p(3000), log_change_1m=math.log(0.85), log_change_3m=math.log(0.65),
        ratio_to_ma3=0.88, drawdown_from_peak=0.25, months_since_release=30.0, next_month=6.0, **genre_flag("Action"),
    ),
    scenario(
        "Juego popular en el top 10 de Twitch",
        log_avg_players=math.log1p(40000), log_change_1m=math.log(1.05), log_change_3m=math.log(1.10),
        ratio_to_ma3=1.03, drawdown_from_peak=0.9, months_since_release=18.0,
        in_twitch_top200=1.0, log_hours_watched=math.log1p(8_000_000), twitch_rank_filled=8.0, **genre_flag("Action"),
    ),
    scenario(
        "Lanzamiento reciente (segundo mes)",
        log_avg_players=math.log1p(5000), log_change_1m=math.log(0.74), has_lag_3m=0.0, log_change_3m=0.0,
        drawdown_from_peak=0.74, months_since_release=1.0, next_month=4.0, **genre_flag("Indie"),
    ),
    scenario(
        "Juego gratuito estable y longevo",
        log_avg_players=math.log1p(15000), log_change_1m=math.log(1.01), log_change_3m=0.0,
        drawdown_from_peak=0.6, months_since_release=96.0, next_month=10.0, is_free=1.0, log_price=0.0,
        **genre_flag("Free To Play"),
    ),
    scenario(
        "Juego pequeño con repunte reciente",
        log_avg_players=math.log1p(60), log_change_1m=math.log(1.4), log_change_3m=math.log(1.8),
        ratio_to_ma3=1.25, drawdown_from_peak=0.5, months_since_release=48.0, next_month=11.0, **genre_flag("Indie"),
    ),
]

scenarios_df = spark.createDataFrame(pd.DataFrame(scenarios))
scenario_results = (
    fitted_models["rf_con_twitch"].transform(scenarios_df)
    .withColumn("probabilidad_crecimiento_pct", F.round(vector_to_array("probability")[1] * 100, 1))
    .select(
        "escenario",
        F.round(F.expm1("log_avg_players"), 0).alias("jugadores_t"),
        F.round(F.exp("log_change_1m") - 1, 3).alias("cambio_1m"),
        "drawdown_from_peak",
        "months_since_release",
        "in_twitch_top200",
        "probabilidad_crecimiento_pct",
        F.when(F.col("prediction") == 1.0, "CRECE ≥ umbral").otherwise("NO CRECE").alias("clasificacion"),
    )
    .toPandas()
)
display(add_report("Escenarios simulados (rf_con_twitch)", scenario_results))

---
## 12. Interpretación de resultados

> **Completar con los resultados de la ejecución.** Citar siempre los números obtenidos.

### Desempeño frente a la referencia
- PR-AUC del mejor modelo vs. referencia vs. azar: _…_
- Recall y precisión de la clase positiva, y qué significan en la práctica (de cada 100 alertas, cuántas aciertan): _…_

### ¿Twitch mejora la predicción? (pregunta de negocio)
- Diferencia de PR-AUC con vs. sin Twitch en todo test: _…_
- Diferencia en las filas con el juego en el top 200: _…_
- Peso de las variables de Twitch en la importancia del modelo: _…_
- Conclusión, como asociación predictiva y no como causalidad: _…_

### Variables que más explican el crecimiento
- _…_

### Aplicaciones en el proyecto
1. **Alerta temprana:** priorizar cada mes los juegos con mayor probabilidad de crecer, por ejemplo para campañas o cobertura editorial.
2. **Evaluación de la exposición en Twitch:** cuantificar si la presencia en el ranking agrega información a la que ya da la dinámica del juego.
3. **Tablero en Databricks SQL** sobre `gold.player_growth_predictions`.

### Limitaciones
- El cruce con Twitch solo usa nombres con coincidencia única; los juegos con nombre ambiguo cuentan como "sin exposición".
- Faltan variables que explican picos: lanzamientos de contenido, actualizaciones, promociones específicas del juego.
- El precio de `dim_game` es el actual, no el de cada mes.
- El umbral de decisión es 0,5 sobre un modelo entrenado con pesos por clase; otro umbral cambiaría el equilibrio entre precisión y recall.

---
## 13. Inferencia por lotes y persistencia en Gold

Se guardan las predicciones de test de los dos Random Forest en `gold.player_growth_predictions`. La escritura reemplaza la tabla completa, así que repetir la ejecución no duplica filas.

In [0]:
if PERSIST:
    frames = []
    for model_name, scored in predictions.items():
        frames.append(
            scored.select(
                "app_id",
                "month_start",
                "name",
                F.col("target").alias("target_real"),
                "probability_growth",
                "prediction",
                F.lit(model_name).alias("model_variant"),
                F.lit(run_ids[model_name]).alias("mlflow_run_id"),
                F.lit(THRESHOLD).alias("growth_threshold"),
            )
        )
    to_save = frames[0]
    for frame in frames[1:]:
        to_save = to_save.unionByName(frame)
    to_save = to_save.withColumn("generated_at", F.current_timestamp())
    (
        to_save.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(gold_table(OUTPUT_TABLE))
    )
    print(f"Tabla guardada: {catalog}.{gold_schema}.{OUTPUT_TABLE} ({to_save.count():,} filas)")
    display(spark.table(gold_table(OUTPUT_TABLE)).orderBy(F.desc("probability_growth")).limit(20))
else:
    print("persist_results=false: no se escribió ninguna tabla.")

---
## 14. Exportación de resultados

Guarda las tablas principales en `notebooks/data_validations/result_outputs/resultados_ml_02_random_forest.md`, dentro del Git Folder, para revisar los resultados fuera de Databricks.

In [0]:
def markdown_value(value):
    if value is None:
        return ""
    if isinstance(value, float):
        if math.isnan(value):
            return ""
        return f"{value:.6g}"
    return str(value).replace("|", "\\|").replace("\n", " ")


def to_markdown(pdf, max_rows=200):
    pdf = pdf.head(max_rows)
    columns = [str(c) for c in pdf.columns]
    lines = ["| " + " | ".join(columns) + " |", "| " + " | ".join(["---"] * len(columns)) + " |"]
    for row in pdf.itertuples(index=False):
        lines.append("| " + " | ".join(markdown_value(v) for v in row) + " |")
    return "\n".join(lines)


report_lines = [
    "# Resultados — Cuaderno ML 2: Random Forest de crecimiento",
    "",
    f"- Generado en UTC: {datetime.now(timezone.utc).isoformat()}",
    f"- Gold: `{catalog}.{gold_schema}`",
    f"- Umbral de crecimiento: {THRESHOLD:.0%} | min_players={MIN_PLAYERS}",
    f"- Ventana: {WINDOW_START} a {WINDOW_END} | train hasta {TRAIN_END}",
    f"- Random Forest: numTrees={NUM_TREES}, maxDepth={MAX_DEPTH}, seed={SEED}",
    f"- Géneros usados: {top_genres}",
    f"- Pesos de clase: positivo={weight_positive:.4f}, negativo={weight_negative:.4f}",
    f"- PR-AUC al azar (tasa de positivos en test): {positive_rate_test:.4f}",
    f"- Diferencia de PR-AUC con Twitch vs. sin Twitch: {pr_gain:+.4f}",
    f"- Importancia total de las variables de Twitch: {twitch_share:.2%}",
    f"- MLflow run ids: {run_ids}",
]
for title, pdf in REPORT:
    report_lines.extend(["", f"## {title}", "", to_markdown(pdf)])
report_text = "\n".join(report_lines) + "\n"

context_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
marker = "/src/ml"
if marker in context_path:
    repo_path = context_path.split(marker, 1)[0]
    workspace_repo_path = repo_path if repo_path.startswith("/Workspace") else "/Workspace" + repo_path
    output_dir = os.path.join(workspace_repo_path, "notebooks", "data_validations", "result_outputs")
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, "resultados_ml_02_random_forest.md")
    with open(output_path + ".tmp", "w", encoding="utf-8", newline="\n") as handle:
        handle.write(report_text)
    os.replace(output_path + ".tmp", output_path)
    print(f"Markdown generado: {output_path}")
else:
    print(f"No se encontró '{marker}' en la ruta {context_path}; se imprime el informe:")
    print(report_text)